# Installations & Imports

In [1]:
!pip install -q scikit-learn

In [21]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np


In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score, classification_report
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.base import clone



from sklearn.preprocessing import LabelEncoder


TRAIN_PATH = "/kaggle/input/datasets/austraa/vashantor/all_dialects_train.csv"
VAL_PATH   = "/kaggle/input/datasets/austraa/vashantor/all_dialects_validation.csv"
TEST_PATH  = "/kaggle/input/datasets/austraa/vashantor/all_dialects_test.csv"

TEXT_COL  = "dialect_speech"
LABEL_COL = "region_name"

ID_COL = None  # change to None if you don't have an id column

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

# Basic column checks
assert TEXT_COL in train_df.columns, f"Missing {TEXT_COL} in train"
assert LABEL_COL in train_df.columns, f"Missing {LABEL_COL} in train"
assert TEXT_COL in val_df.columns, f"Missing {TEXT_COL} in validation"
assert LABEL_COL in val_df.columns, f"Missing {LABEL_COL} in validation"
assert TEXT_COL in test_df.columns, f"Missing {TEXT_COL} in test"

# Preparing Data

In [3]:
def prep_text(s: pd.Series) -> pd.Series:
    s = s.fillna("").astype(str)
    # whitespace cleanup 
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

train_df[TEXT_COL] = prep_text(train_df[TEXT_COL])
val_df[TEXT_COL]   = prep_text(val_df[TEXT_COL])
test_df[TEXT_COL]  = prep_text(test_df[TEXT_COL])

train_df = train_df[(train_df[TEXT_COL].str.len() > 0) & (train_df[LABEL_COL].astype(str).str.len() > 0)].reset_index(drop=True)
val_df   = val_df[(val_df[TEXT_COL].str.len() > 0) & (val_df[LABEL_COL].astype(str).str.len() > 0)].reset_index(drop=True)

In [4]:
# Encode labels

def norm_label(s: pd.Series) -> pd.Series:
    s = s.fillna("").astype(str)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()   # removes trailing/extra spaces
    return s

train_df[LABEL_COL] = norm_label(train_df[LABEL_COL])
val_df[LABEL_COL]   = norm_label(val_df[LABEL_COL])

# encode
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL].values)
y_val   = le.transform(val_df[LABEL_COL].values)

X_train = train_df[TEXT_COL].values
X_val   = val_df[TEXT_COL].values
X_test  = test_df[TEXT_COL].values

# Model Building

In [5]:
# Features

word_tfidf = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2),
    min_df=2, max_df=0.95, sublinear_tf=True
)

char_tfidf = TfidfVectorizer(
    analyzer="char", ngram_range=(3, 6),
    min_df=2, max_df=0.95, sublinear_tf=True
)

charwb_tfidf = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 6),
    min_df=2, max_df=0.95, sublinear_tf=True
)

features = FeatureUnion([
    ("word", word_tfidf),
    ("char", char_tfidf),
    ("charwb", charwb_tfidf),
])

In [9]:
# Classifiers

candidates = {
    "LogReg_balanced_C4": LogisticRegression(
        max_iter=3000, n_jobs=-1, C=4.0,
        solver="lbfgs", multi_class="auto",
        class_weight="balanced"
    ),
    "LinearSVC_C1": LinearSVC(C=1.0),
    "SGD_hinge_a1e-5": SGDClassifier(
        loss="hinge", alpha=1e-5, max_iter=2000, tol=1e-3,
        random_state=42
    ),
}




In [19]:

results = []
trained_models = {}

param_grids = {
    "LogReg_balanced_C4": {
        "features__word__ngram_range": [(1,2), (1,3)],
        "features__char__ngram_range": [(3,6), (2,6), (3,7)],
        "features__charwb__ngram_range": [(3,6), (4,7)],
        "clf__C": [0.5, 1.0, 2.0, 4.0],
    },
    "LinearSVC_C1": {
        "features__word__ngram_range": [(1,2), (1,3)],
        "features__char__ngram_range": [(3,6), (2,6), (3,7)],
        "features__charwb__ngram_range": [(3,6), (4,7)],
        "clf__C": [0.5, 1.0, 2.0, 4.0],
    },
    "SGD_hinge_a1e-5": {
        "features__word__ngram_range": [(1,2), (1,3)],
        "features__char__ngram_range": [(3,6), (2,6), (3,7)],
        "features__charwb__ngram_range": [(3,6), (4,7)],
        "clf__alpha": [1e-6, 3e-6, 1e-5, 3e-5],   # <- SGD uses alpha, not C
    }
}

for name, clf in candidates.items():
    model = Pipeline([
        ("features", features),
        ("clf", clf),
    ])

    grid = param_grids[name]  # must match the keys in candidates

    gs = GridSearchCV(
        estimator=model,
        param_grid=grid,
        scoring="accuracy",     # consider "f1_macro" if imbalance matters
        n_jobs=-1,
        verbose=1
    )
    gs.fit(X_train, y_train)

    best_model = gs.best_estimator_   # already refit on full train
    val_pred = best_model.predict(X_val)

    acc = accuracy_score(y_val, val_pred)
    macro_f1 = f1_score(y_val, val_pred, average="macro")

    results.append({
        "model": name,
        "best_cv_score": gs.best_score_,
        "val_acc": acc,
        "val_macro_f1": macro_f1,
        "best_params": gs.best_params_
    })
    trained_models[name] = best_model

res_df = pd.DataFrame(results).sort_values(["val_macro_f1", "val_acc"], ascending=False)
display(res_df[["model", "best_cv_score", "val_acc", "val_macro_f1"]])

best_name = res_df.iloc[0]["model"]
best_model = trained_models[best_name]
print("\nBest model:", best_name)
print("\nBest params:\n", res_df.iloc[0]["best_params"])

print("\nValidation report for best model:\n")
print(classification_report(y_val, best_model.predict(X_val), target_names=le.classes_))

Fitting 5 folds for each of 48 candidates, totalling 240 fits


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

Fitting 5 folds for each of 48 candidates, totalling 240 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Fitting 5 folds for each of 48 candidates, totalling 240 fits


,model,best_cv_score,val_acc,val_macro_f1
1,LinearSVC_C1,0.901867,0.9072,0.906673
0,LogReg_balanced_C4,0.902613,0.9048,0.904250
2,SGD_hinge_a1e-5,0.896213,0.9000,0.899461



Best model: LinearSVC_C1

Best params:
 {'clf__C': 0.5, 'features__char__ngram_range': (2, 6), 'features__charwb__ngram_range': (3, 6), 'features__word__ngram_range': (1, 3)}

Validation report for best model:

              precision    recall  f1-score   support

    Barishal       0.93      0.95      0.94       250
  Chittagong       0.90      0.98      0.93       250
  Mymensingh       0.86      0.89      0.87       250
    Noakhali       0.93      0.89      0.91       250
      Sylhet       0.92      0.83      0.88       250

    accuracy                           0.91      1250
   macro avg       0.91      0.91      0.91      1250
weighted avg       0.91      0.91      0.91      1250



In [22]:

X_final = np.concatenate([X_train, X_val], axis=0)
y_final = np.concatenate([y_train, y_val], axis=0)

# best_final = clone(trained_models[best_name])
# best_final.fit(X_final, y_final)
best_final = clone(trained_models[best_name])
best_final.set_params(**res_df.iloc[0]["best_params"])
best_final.fit(X_final, y_final)

test_pred_ids = best_final.predict(X_test)
test_pred_labels = le.inverse_transform(test_pred_ids)

def norm_label(s: pd.Series) -> pd.Series:
    s = s.fillna("").astype(str)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

test_true = norm_label(test_df[LABEL_COL])
test_pred = norm_label(pd.Series(test_pred_labels))

test_acc = accuracy_score(test_true, test_pred)
print("\nTEST Accuracy (best tuned retrained):", test_acc)

tmp = pd.DataFrame({"true": test_true.values, "pred": test_pred.values})
region_stats = (
    tmp.assign(correct=(tmp["true"] == tmp["pred"]).astype(int))
       .groupby("true")
       .agg(
           support=("correct","size"),
           correct=("correct","sum"),
           accuracy=("correct","mean")
       )
       .sort_values(["accuracy","support"], ascending=[False, False])
)
display(region_stats)

print("\nTEST classification report:\n")
print(classification_report(tmp["true"], tmp["pred"], labels=le.classes_))

cm = confusion_matrix(tmp["true"], tmp["pred"], labels=le.classes_)
cm_df = pd.DataFrame(
    cm,
    index=[f"true:{c}" for c in le.classes_],
    columns=[f"pred:{c}" for c in le.classes_]
)
display(cm_df)


TEST Accuracy (best tuned retrained): 0.8650666666666667


,support,correct,accuracy
true,,,
Mymensingh,375,367,0.978667
Barishal,375,365,0.973333
Chittagong,375,345,0.920000
Noakhali,375,303,0.808000
Sylhet,375,242,0.645333



TEST classification report:

              precision    recall  f1-score   support

    Barishal       0.96      0.97      0.96       375
  Chittagong       0.88      0.92      0.90       375
  Mymensingh       0.75      0.98      0.85       375
    Noakhali       0.85      0.81      0.83       375
      Sylhet       0.94      0.65      0.76       375

    accuracy                           0.87      1875
   macro avg       0.88      0.87      0.86      1875
weighted avg       0.88      0.87      0.86      1875



,pred:Barishal,pred:Chittagong,pred:Mymensingh,pred:Noakhali,pred:Sylhet
true:Barishal,365,0,7,2,1
true:Chittagong,1,345,0,26,3
true:Mymensingh,2,0,367,3,3
true:Noakhali,3,39,21,303,9
true:Sylhet,11,9,92,21,242


# Majority Voting

In [23]:
lr_model = Pipeline([
    ("features", features),
    ("clf", LogisticRegression(
        max_iter=3000, n_jobs=-1, C=4.0, solver="lbfgs",
        multi_class="auto", class_weight="balanced"
    ))
])

svc_model = Pipeline([
    ("features", features),
    ("clf", LinearSVC(C=1.0))
])

sgd_model = Pipeline([
    ("features", features),
    ("clf", SGDClassifier(
        loss="hinge", alpha=1e-5, max_iter=2000, tol=1e-3,
        random_state=42
    ))
])

# Train
lr_model.fit(X_train, y_train)
svc_model.fit(X_train, y_train)
sgd_model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word',
                                                 TfidfVectorizer(max_df=0.95,
                                                                 min_df=2,
                                                                 ngram_range=(1,
                                                                              2),
                                                                 sublinear_tf=True)),
                                                ('char',
                                                 TfidfVectorizer(analyzer='char',
                                                                 max_df=0.95,
                                                                 min_df=2,
                                                                 ngram_range=(3,
                                                                              6),
                                                                 sublinear_tf=True)),
                                                ('charwb',
                                                 TfidfVectorizer(analyzer='char_wb',
                                                                 max_df=0.95,
                                                                 min_df=2,
                                                                 ngram_range=(3,
                                                                              6),
                                                                 sublinear_tf=True))])),
                ('clf',
                 SGDClassifier(alpha=1e-05, max_iter=2000, random_state=42))])

In [29]:
def weighted_vote(preds_2d: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """
    preds_2d: (n_models, n_samples) integer ids
    weights: (n_models,) weights
    """
    n_models, n_samples = preds_2d.shape
    out = np.empty(n_samples, dtype=preds_2d.dtype)

    for i in range(n_samples):
        score = {}
        for m in range(n_models):
            cls = int(preds_2d[m, i])
            score[cls] = score.get(cls, 0.0) + float(weights[m])
        out[i] = max(score.items(), key=lambda kv: kv[1])[0]
    return out

weights = np.array([4.0, 3.0, 3.0])

# val_wens = weighted_vote(preds, weights)
# print("Weighted Ensemble Val Accuracy:", accuracy_score(y_val, val_wens))
# print("Weighted Ensemble Val Macro-F1:", f1_score(y_val, val_wens, average="macro"))

In [30]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Get predictions (ids)
p_lr  = lr_model.predict(X_val)
p_svc = svc_model.predict(X_val)
p_sgd = sgd_model.predict(X_val)

preds = np.vstack([p_lr, p_svc, p_sgd])  # (3, n)

val_ens = weighted_vote(preds, weights)

print("Ensemble Val Accuracy:", accuracy_score(y_val, val_ens))
print("Ensemble Val Macro-F1:", f1_score(y_val, val_ens, average="macro"))
print("\nClassification report (ensemble):\n")
print(classification_report(y_val, val_ens, target_names=le.classes_))

Ensemble Val Accuracy: 0.996
Ensemble Val Macro-F1: 0.9960031648891535

Classification report (ensemble):

              precision    recall  f1-score   support

    Barishal       1.00      1.00      1.00       250
  Chittagong       1.00      1.00      1.00       250
  Mymensingh       0.99      1.00      0.99       250
    Noakhali       1.00      1.00      1.00       250
      Sylhet       1.00      0.99      0.99       250

    accuracy                           1.00      1250
   macro avg       1.00      1.00      1.00      1250
weighted avg       1.00      1.00      1.00      1250



In [31]:
# Retrain each model on train+val
X_final = np.concatenate([X_train, X_val], axis=0)
y_final = np.concatenate([y_train, y_val], axis=0)

lr_final  = lr_model.fit(X_final, y_final)
svc_final = svc_model.fit(X_final, y_final)
sgd_final = sgd_model.fit(X_final, y_final)

# Predict on test
pt_lr  = lr_final.predict(X_test)
pt_svc = svc_final.predict(X_test)
pt_sgd = sgd_final.predict(X_test)

test_preds = np.vstack([pt_lr, pt_svc, pt_sgd])
test_ens_ids = weighted_vote(test_preds, weights)  # or weighted_vote(test_preds, weights)

# ids -> labels
test_pred_labels = le.inverse_transform(test_ens_ids)

test_true = norm_label(test_df[LABEL_COL])
test_pred = norm_label(pd.Series(test_pred_labels))

test_acc = accuracy_score(test_true, test_pred)
print("\nTEST Accuracy (best retrained):", test_acc)

# Region-wise accuracy table
tmp = pd.DataFrame({"true": test_true.values, "pred": test_pred.values})
region_stats = (
    tmp.assign(correct=(tmp["true"] == tmp["pred"]).astype(int))
       .groupby("true")
       .agg(support=("correct","size"),
            correct=("correct","sum"),
            accuracy=("correct","mean"))
       .sort_values(["accuracy","support"], ascending=[False, False])
)
display(region_stats)

print("\nTEST classification report:\n")
print(classification_report(tmp["true"], tmp["pred"], labels=le.classes_))

cm = confusion_matrix(tmp["true"], tmp["pred"], labels=le.classes_)
cm_df = pd.DataFrame(cm, index=[f"true:{c}" for c in le.classes_],
                        columns=[f"pred:{c}" for c in le.classes_])
display(cm_df)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



TEST Accuracy (best retrained): 0.8677333333333334


,support,correct,accuracy
true,,,
Mymensingh,375,369,0.984000
Barishal,375,366,0.976000
Chittagong,375,343,0.914667
Noakhali,375,303,0.808000
Sylhet,375,246,0.656000



TEST classification report:

              precision    recall  f1-score   support

    Barishal       0.96      0.98      0.97       375
  Chittagong       0.89      0.91      0.90       375
  Mymensingh       0.76      0.98      0.86       375
    Noakhali       0.86      0.81      0.83       375
      Sylhet       0.92      0.66      0.77       375

    accuracy                           0.87      1875
   macro avg       0.88      0.87      0.86      1875
weighted avg       0.88      0.87      0.86      1875



,pred:Barishal,pred:Chittagong,pred:Mymensingh,pred:Noakhali,pred:Sylhet
true:Barishal,366,0,6,2,1
true:Chittagong,1,343,0,26,5
true:Mymensingh,2,0,369,1,3
true:Noakhali,3,37,21,303,11
true:Sylhet,11,7,89,22,246
